## **Projeto:** Merca Data Platform
### **Squad:** 2 | Camada Bronze — ecommerce_categorias
### Origem e Destino
| Item | Valor |

| **Origem** | `real-time-data/<snapshot>/ecommerce_categorias.parquet` |

| **Destino** | `squad2/bronze/ecommerce_categorias` (Delta Lake) |

| **Checkpoint** | `squad2/control/bronze/ecommerce_categorias/control_file.json` |

| **Polling** | Verifica novos snapshots a cada 30 segundos |

### Colunas de Auditoria
| Coluna | Descrição |

| `_snapshot_id` | ID do snapshot (`YYYY/MM/DD/HHMMSS`) |

| `_ingested_at` | Timestamp de ingestão |

| `_source` | Fonte dos dados (`real-time-data`) |

| `_camada` | Camada atual (`bronze`) |

In [0]:
%pip install deltalake

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
import logging
from pyspark.sql.functions import lit, current_timestamp, year, month, dayofmonth, hour

logging.getLogger("azure").setLevel(logging.WARNING)

TABELA = "ecommerce_categorias"
CAMADA = "bronze"

inicio = log_inicio("feat_squad2_" + CAMADA + "_" + TABELA)
log.info("Tabela : " + TABELA)
log.info("Camada : " + CAMADA)
log.info("Path   : " + get_delta_path(CAMADA, TABELA))

- Listagem de Snapshots Disponíveis
Lista todos os pacotes de dados disponíveis no container raw.
Os snapshots são exibidos em ordem cronológica para facilitar a auditoria.

In [0]:
def processar_snapshot(snapshot_id: str) -> bool:
    try:
        df = ler_parquet(snapshot_id, TABELA)

        source_file = "real-time-data/" + snapshot_id + "/" + TABELA + ".parquet"

        df_bronze = df \
            .withColumn("bronze_source_file", lit(source_file)) \
            .withColumn("bronze_ingested_at", current_timestamp()) \
            .withColumn("_source",            lit("real-time-data")) \
            .withColumn("_camada",            lit(CAMADA)) \
            .withColumn("ingestion_year",     year(current_timestamp()).cast("string")) \
            .withColumn("ingestion_month",    month(current_timestamp()).cast("string")) \
            .withColumn("ingestion_day",      dayofmonth(current_timestamp()).cast("string")) \
            .withColumn("ingestion_hour",     hour(current_timestamp()).cast("string"))

        sucesso = gravar_delta(df_bronze, CAMADA, TABELA)

        if sucesso:
            log.info("OK " + snapshot_id + " -> " + str(df_bronze.count()) + " linhas gravadas.")

        return sucesso

    except Exception as e:
        log.error("Erro ao processar " + snapshot_id + ": " + str(e))
        return False

 %md
### Validação Pontual
Execute esta célula isoladamente para verificar o estado atual sem iniciar o loop contínuo.

In [0]:
try:
    snapshots   = sorted(listar_snapshots())
    processados = ler_checkpoint(CAMADA, TABELA)
    novos       = [s for s in snapshots if s not in processados]

    log.info(f"Snapshots disponíveis : {len(snapshots)}")
    log.info(f"Já processados        : {len(processados)}")
    log.info(f"Novos para processar  : {len(novos)}")
    log.info(f"Status atual          : {ler_status_checkpoint(CAMADA, TABELA)}")

    if not novos:
        log.info(" Bronze categorias em dia!")
    else:
        for s in novos:
            print(f" Processados {s}")

except Exception as e:
    log.error(f"Erro na validação: {str(e)}")
    raise

### Polling Loop
Inicia o monitoramento contínuo da Bronze.

**Bronze não tem dependência de camada anterior.**

Para encerrar, interrompa a execução manualmente.

In [0]:
snapshots   = sorted(listar_snapshots())
processados = ler_checkpoint(CAMADA, TABELA)
novos       = [s for s in snapshots if s not in processados]

if not novos:
    log.info("Bronze " + TABELA + " em dia - nenhum snapshot novo.")
else:
    log.info(str(len(novos)) + " snapshot(s) novo(s) encontrado(s).")
    salvar_checkpoint(CAMADA, TABELA, processados, status="PROCESSANDO")

    for snapshot_id in novos:
        log.info("Processando: " + snapshot_id)
        sucesso = processar_snapshot(snapshot_id)
        if sucesso:
            processados.add(snapshot_id)
            log.info("OK: " + snapshot_id)
        else:
            log.warning("FALHOU: " + snapshot_id + " - sera retentado no proximo ciclo.")

    salvar_checkpoint(CAMADA, TABELA, processados, status="CONCLUIDO")
    log.info("Bronze " + TABELA + " concluida.")

log_fim("feat_squad2_" + CAMADA + "_" + TABELA, inicio)